# 实验 010：锚点增强融合（特征增强 × 时间策略 × 稳定性定权）

`exp_010` 合并 9 个历史实验的优点，以正式提交 `prediction.npy` 为只读锚点，仅训练并融合"近期专家"：

- 主特征沿用 `legacy_328`（exp_003），增量候选为 `exp_008` 数值特征银行的 EWMA surprise（半衰期 5/20/60）+ trend（fast/slow）+ history_state 块（合计 105 维），先做 3 折诊断、全部折为正且均值增量 ≥ 0.0005 才并入，否则回退 328；
- 专家训练历史候选 `full`（近期 1702 期、均匀权重）vs `decay_1200`（近期窗口 + 半衰期 1200 指数时间衰减），在 3 折内比较均值与后段后二选一；
- 轮数扫 8/16/32，权重网格 {0.00 … 0.35}，在最佳平均增益 95% 区间内取最低权重，并叠加 exp_004 稳定性平台（±0.1 邻近权重全窗波动 < 0.001）；
- 官方 Valid 只做一次性晋级检查，绝不参与搜索；正式提交文件绝不覆盖。


## tl;dr

- 默认 `DSCR_EXP010_MODE=full`：依次执行 exp_009 基线复现、特征诊断、时间策略对比、轮数/权重选择、官方 Valid 一次性检查、最终专家重训与完整 Test 预测。
- 每次完整运行都会生成 `prediction.npy`、`expert_prediction.npy`、`valid_prediction.npy`、`metrics.json`、`metadata.json`、`experiment_report.md` 及全部 CSV 明细；晋级时额外生成 `promoted_candidate.npy`。
- 如果没有任何候选通过稳定性门槛，权重回退到 `0`，输出与正式锚点一致的预测文件。
- `DSCR_EXP010_MODE=preflight` 只做数据契约（含 exp_008 特征银行行对齐）、轻量冒烟与写出逻辑验证，产物写入 `04_results/exp_010_anchor_enhanced_blend/preflight/`。
- TCN 低权重差异信号为可选（`DSCR_EXP010_USE_TCN=1`），默认关闭。


## Context & Methods

### Key Assumptions

1. `processed_data_v1` 的 `legacy_328` 与 `exp_003` 稳定特征视图兼容，且 READY、manifest SHA-256、legacy compatibility 均通过。
2. `exp_008` 数值特征银行（363 维）的行序与 `processed_data_v1` common 面板逐行一致；增量块只取 surprise/trend/history_state（105 维），不含原始值与截面排名。
3. 所有训练、特征诊断、时间策略、权重与轮数选择只使用 Train 内时间折；官方 Valid 只做一次晋级检查，不反向修改权重。
4. Test 锚点直接读取当前正式提交；本实验只训练专家，绝不重训或覆盖锚点。
5. 每个时间截面先独立转百分位秩再融合，非评价位置严格填充 `0.5`。


In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata

EXPECTED_CONDA_ENV = "jingge_ts"
CURRENT_CONDA_ENV = Path(sys.prefix).name
if CURRENT_CONDA_ENV.lower() != EXPECTED_CONDA_ENV.lower():
    raise RuntimeError(
        "请把 Notebook 内核切换到 Anaconda 环境 " + EXPECTED_CONDA_ENV
        + "；当前解释器为 " + sys.executable
    )


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data.z").exists() and (candidate / "02_experiments").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录：请从项目根目录或实验目录启动 Notebook。")


@dataclass(frozen=True)
class WalkForwardFold:
    name: str
    train_start: int
    train_stop: int
    valid_start: int
    valid_stop: int


PROJECT_ROOT = find_project_root()
EXPERIMENT_ID = "exp_010_anchor_enhanced_blend"
DATASET_DIR = PROJECT_ROOT / "03_cache" / "processed_data_v1"
EXP008_BANK_ROOT = (
    PROJECT_ROOT / "03_cache" / "exp_008_new_method" / "numeric_feature_bank_v2" / "7607a1f17073"
)
OUTPUT_DIR = PROJECT_ROOT / "04_results" / EXPERIMENT_ID
MANIFEST_PATH = DATASET_DIR / "manifest.json"
READY_PATH = DATASET_DIR / "READY"
ANCHOR_PATH = PROJECT_ROOT / "04_results" / "final_submission" / "prediction.npy"
ANCHOR_METADATA_PATH = PROJECT_ROOT / "04_results" / "final_submission" / "metadata.json"

RUN_MODE = os.environ.get("DSCR_EXP010_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "preflight"}:
    raise ValueError("DSCR_EXP010_MODE 只允许 full 或 preflight。")
RUN_DIR = OUTPUT_DIR if RUN_MODE == "full" else OUTPUT_DIR / "preflight"
RUNTIME_CACHE_DIR = RUN_DIR / "runtime_cache"

# ---- 全局数据切分 ----
TRAIN_START, TRAIN_STOP = 486, 2918
VALID_START, VALID_STOP = 2918, 3161
TEST_START, TEST_STOP = 3161, 3603
TEST_TIME_POINTS, STOCK_COUNT = 442, 5282

# ---- 特征配置：legacy_328 + exp_008 增量块（surprise + trend + history_state）----
FEATURE_STOP = 328
INCR_COLS = np.concatenate(
    [np.arange(198, 258), np.arange(318, 363)]
).astype(np.int64)
INCR_SIZE = int(INCR_COLS.size)
ENHANCED_FEATURE_COUNT = FEATURE_STOP + INCR_SIZE

# ---- 训练与选择配置 ----
TRAIN_STOCK_CAP = 1200
RECENT_LOOKBACK = 1702
BASE_ROUNDS = 8
ROUNDS_CANDIDATES = (8, 16, 32)
RECENT_WEIGHTS = (0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35)
DECAY_HALF_LIFE = 1200.0
STRATEGIES = ("full", "decay_1200")
SEED = 42
NUM_THREADS = max(1, (os.cpu_count() or 8) - 2)
MAX_EXPERT_ROUNDS = max(ROUNDS_CANDIDATES)

# ---- 特征诊断 / 时间策略 / 选择门槛 ----
DIAGNOSIS_MIN_MEAN_GAIN = 0.0005
MIN_POSITIVE_FOLDS = 2
MIN_MEAN_IMPROVEMENT = 0.0005
MAX_WORST_FOLD_DROP = 0.0015
MAX_LATE_MEAN_DROP = 0.0003
PLATEAU_WINDOW = 0.10
PLATEAU_MAX_SPAN = 0.001
OFFICIAL_MIN_IMPROVEMENT = 0.0003
OFFICIAL_MAX_LATE_DROP = 0.0002
OFFICIAL_MAX_WORST_QUARTER_DROP = 0.0015
ANCHOR_EXPECTED_VALID_IC = 0.09294016824452567
ANCHOR_REPRODUCTION_TOLERANCE = 0.0003
MIN_TEST_ANCHOR_CORRELATION = 0.97

# ---- exp_009 复现目标（fold 内锚点 8 轮 与 (16, 0.25) 融合）----
EXP009_FOLD_TARGETS = {
    "fold_1": {"anchor": 0.11203110472049536, "blend16_25": 0.1130180669902052},
    "fold_2": {"anchor": 0.0971174118037759, "blend16_25": 0.09800554027556392},
    "fold_3": {"anchor": 0.08377222907415181, "blend16_25": 0.08506900620216068},
}
REPRODUCTION_TOLERANCE = 5e-4

# ---- 可选 TCN 低权重差异信号 ----
USE_TCN = int(os.environ.get("DSCR_EXP010_USE_TCN", "0"))
if USE_TCN not in {0, 1}:
    raise ValueError("DSCR_EXP010_USE_TCN 只允许 0 或 1。")
TCN_WEIGHTS = (0.00, 0.05, 0.10, 0.15)
MAX_TCN_WEIGHT = 0.15

# ---- 走步折（与 exp_009 一致）；preflight 用极短折 ----
FOLDS = (
    WalkForwardFold("fold_1", 486, 2189, 2189, 2432),
    WalkForwardFold("fold_2", 486, 2432, 2432, 2675),
    WalkForwardFold("fold_3", 486, 2675, 2675, 2918),
)
PREFLIGHT_FOLDS = (
    WalkForwardFold("fold_1", 486, 560, 560, 570),
    WalkForwardFold("fold_2", 486, 570, 570, 580),
    WalkForwardFold("fold_3", 486, 580, 580, 590),
)

random.seed(SEED)
np.random.seed(SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(pd.Series({
    "project_root": str(PROJECT_ROOT),
    "run_mode": RUN_MODE,
    "run_dir": str(RUN_DIR),
    "base_features": FEATURE_STOP,
    "incremental_features": INCR_SIZE,
    "enhanced_features": ENHANCED_FEATURE_COUNT,
    "stock_cap": TRAIN_STOCK_CAP,
    "recent_lookback": RECENT_LOOKBACK,
    "rounds_candidates": ROUNDS_CANDIDATES,
    "weights": RECENT_WEIGHTS,
    "strategies": STRATEGIES,
    "use_tcn": USE_TCN,
    "threads": NUM_THREADS,
}))


project_root                                        D:\google_dl\book\友安杯
run_mode                                                             full
run_dir                 D:\google_dl\book\友安杯\04_results\exp_010_ancho...
base_features                                                         328
incremental_features                                                  105
enhanced_features                                                     433
stock_cap                                                            1200
recent_lookback                                                      1702
rounds_candidates                                             (8, 16, 32)
weights                      (0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35)
strategies                                             (full, decay_1200)
use_tcn                                                                 0
threads                                                                30
dtype: object


## Data

### 1. 验证重型缓存并加载固定视图（含 exp_008 特征银行行对齐）

In [2]:
def file_sha256(path: Path, block_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    partial.write_text(text, encoding="utf-8")
    os.replace(partial, path)


def atomic_write_json(path: Path, payload: dict) -> None:
    atomic_write_text(path, json.dumps(payload, ensure_ascii=False, indent=2))


def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    with partial.open("wb") as handle:
        np.save(handle, array)
    os.replace(partial, path)


def atomic_save_npz(path: Path, **arrays: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    with partial.open("wb") as handle:
        np.savez(handle, **arrays)
    os.replace(partial, path)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".partial")
    frame.to_csv(partial, index=False, encoding="utf-8-sig")
    os.replace(partial, path)


if not READY_PATH.exists() or not MANIFEST_PATH.exists():
    raise RuntimeError("processed_data_v1 缺少 READY 或 manifest.json，禁止训练。")

ready = json.loads(READY_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_sha256 = file_sha256(MANIFEST_PATH)

assert manifest["status"] == "ready"
assert ready["manifest_sha256"] == manifest_sha256
assert manifest["dimensions"] == {"time": 3603, "stock": 5282, "raw_numeric": 99, "raw_category": 9}
assert manifest["splits"]["train"]["start"] == TRAIN_START
assert manifest["splits"]["train"]["stop"] == TRAIN_STOP
assert manifest["splits"]["valid"]["start"] == VALID_START
assert manifest["splits"]["valid"]["stop"] == VALID_STOP
assert manifest["splits"]["test"]["start"] == TEST_START
assert manifest["splits"]["test"]["stop"] == TEST_STOP
assert manifest["features"]["legacy_numeric_prefix"] == FEATURE_STOP
assert manifest["features"]["tree_count"] == 419
assert manifest["legacy_compatibility"]["status"] == "passed"
assert manifest["validation"]["test_mask"]["count"] == 2_042_538
assert manifest["validation"]["test_mask"]["matches_official"] is True


def load_common(split: str) -> dict:
    directory = DATASET_DIR / "common"
    result = {
        "time": np.load(directory / f"{split}_time.npy", mmap_mode="r"),
        "stock": np.load(directory / f"{split}_stock.npy", mmap_mode="r"),
        "groups": np.load(directory / f"{split}_group_sizes.npy", mmap_mode="r"),
    }
    if split != "test":
        result["y"] = np.load(directory / f"{split}_y.npy", mmap_mode="r")
        result["relevance"] = np.load(directory / f"{split}_relevance.npy", mmap_mode="r")
    return result


def load_tree(split: str) -> np.ndarray:
    matrix = np.load(DATASET_DIR / "tree" / f"{split}_X.npy", mmap_mode="r")
    expected_rows = int(manifest["expected_rows"][split])
    assert matrix.shape == (expected_rows, 419)
    return matrix


common = {split: load_common(split) for split in ("train", "valid", "test")}
tree = {split: load_tree(split) for split in ("train", "valid", "test")}

data_rows = []
for split in ("train", "valid", "test"):
    values = common[split]
    assert int(values["groups"].sum()) == values["time"].size == values["stock"].size
    assert np.all(np.diff(values["time"]) >= 0)
    assert np.isfinite(tree[split][:32, :FEATURE_STOP]).all()
    data_rows.append({
        "split": split,
        "rows": int(values["time"].size),
        "time_start": int(values["time"][0]),
        "time_stop": int(values["time"][-1]) + 1,
        "time_points": int(values["groups"].size),
        "group_sum": int(values["groups"].sum()),
    })

assert ANCHOR_PATH.exists() and ANCHOR_METADATA_PATH.exists()
anchor_metadata = json.loads(ANCHOR_METADATA_PATH.read_text(encoding="utf-8"))
assert anchor_metadata["prediction_sha256"] == file_sha256(ANCHOR_PATH)

# ---- exp_008 数值特征银行：结构与行对齐 ----
bank_meta = {}
bank_time = {}
bank_stock = {}
bank_groups = {}
BANK_MATRIX = {}
for role in ("train_valid", "test"):
    role_dir = EXP008_BANK_ROOT / role
    meta = json.loads((role_dir / "metadata.json").read_text(encoding="utf-8"))
    assert meta["feature_count"] == 363
    assert meta["feature_blocks"]["surprise"] == [198, 258]
    assert meta["feature_blocks"]["trend"] == [318, 358]
    assert meta["feature_blocks"]["history_state"] == [358, 363]
    bank_meta[role] = meta
    bank_time[role] = np.load(role_dir / "time_index.npy")
    bank_stock[role] = np.load(role_dir / "stock_index.npy")
    bank_groups[role] = np.load(role_dir / "group_sizes.npy")
    BANK_MATRIX[role] = np.load(role_dir / "features.npy", mmap_mode="r")
    assert bank_time[role].size == int(meta["rows"])

train_rows = int(common["train"]["time"].size)
valid_rows = int(common["valid"]["time"].size)
assert np.array_equal(
    bank_time["train_valid"],
    np.concatenate([np.asarray(common["train"]["time"]), np.asarray(common["valid"]["time"])]),
)
assert np.array_equal(
    bank_stock["train_valid"],
    np.concatenate([np.asarray(common["train"]["stock"]), np.asarray(common["valid"]["stock"])]),
)
assert np.array_equal(
    bank_groups["train_valid"],
    np.concatenate([np.asarray(common["train"]["groups"]), np.asarray(common["valid"]["groups"])]),
)
assert np.array_equal(bank_time["test"], np.asarray(common["test"]["time"]))
assert np.array_equal(bank_stock["test"], np.asarray(common["test"]["stock"]))
assert np.array_equal(bank_groups["test"], np.asarray(common["test"]["groups"]))

BANK_OFFSET = {"train": 0, "valid": train_rows, "test": 0}


def bank_incremental(split: str, indices: np.ndarray) -> np.ndarray:
    indices = np.asarray(indices, dtype=np.int64)
    matrix = BANK_MATRIX["test"] if split == "test" else BANK_MATRIX["train_valid"]
    return np.asarray(matrix[indices + BANK_OFFSET[split]][:, INCR_COLS], dtype=np.float32)


assert INCR_SIZE == 105
assert ENHANCED_FEATURE_COUNT == 433
sample_idx = np.linspace(0, 2000, 32, dtype=np.int64)
assert np.isfinite(bank_incremental("train", sample_idx)).all()
assert np.isfinite(bank_incremental("test", sample_idx)).all()

DATA_CONTRACT = pd.DataFrame(data_rows)
display(DATA_CONTRACT)
print("缓存、正式锚点与 exp_008 特征银行（" + str(INCR_SIZE) + " 维增量块）数据契约：通过。")


,split,rows,time_start,time_stop,time_points,group_sum
0,train,6489099,486,2918,2432,6489099
1,valid,982972,2918,3161,243,982972
2,test,2042538,3161,3603,442,2042538


缓存、正式锚点与 exp_008 特征银行（105 维增量块）数据契约：通过。


### 2. 指标、时间切片和截面秩工具

In [3]:
def rank_ic(prediction: np.ndarray, target: np.ndarray) -> float:
    prediction = np.asarray(prediction)
    target = np.asarray(target)
    usable = np.isfinite(prediction) & np.isfinite(target)
    if int(usable.sum()) < 2:
        return np.nan
    x = rankdata(prediction[usable])
    y = rankdata(target[usable])
    if x.std() == 0 or y.std() == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def group_rank_ic_series(prediction: np.ndarray, target: np.ndarray, groups: np.ndarray) -> np.ndarray:
    prediction = np.asarray(prediction)
    target = np.asarray(target)
    values = []
    offset = 0
    for size in groups:
        size = int(size)
        values.append(rank_ic(prediction[offset:offset + size], target[offset:offset + size]))
        offset += size
    assert offset == prediction.size == target.size
    return np.asarray(values, dtype=np.float64)


def group_rank_transform(prediction: np.ndarray, groups: np.ndarray) -> np.ndarray:
    prediction = np.asarray(prediction)
    ranked = np.empty(prediction.size, dtype=np.float32)
    offset = 0
    for size in groups:
        size = int(size)
        ranked[offset:offset + size] = rankdata(
            prediction[offset:offset + size], method="average"
        ).astype(np.float32) / float(size)
        offset += size
    assert offset == prediction.size
    return ranked


def score_prediction(prediction: np.ndarray, target: np.ndarray, groups: np.ndarray) -> dict:
    per_time = group_rank_ic_series(prediction, target, groups)
    quarters = np.array_split(per_time, 4)
    return {
        "mean_rank_ic": float(np.nanmean(per_time)),
        "std_rank_ic": float(np.nanstd(per_time)),
        "late_half_rank_ic": float(np.nanmean(per_time[len(per_time) // 2:])),
        "worst_quarter_rank_ic": float(min(np.nanmean(part) for part in quarters)),
        "negative_time_share": float(np.mean(per_time < 0)),
    }


def row_slice_for_times(split: str, start_time: int, stop_time: int) -> tuple:
    times = common[split]["time"]
    start = int(np.searchsorted(times, start_time, side="left"))
    stop = int(np.searchsorted(times, stop_time, side="left"))
    split_start = int(times[0])
    group_start = start_time - split_start
    group_stop = stop_time - split_start
    groups = np.asarray(common[split]["groups"][group_start:group_stop], dtype=np.int32)
    assert int(groups.sum()) == stop - start
    return slice(start, stop), groups


def capped_indices_for_split(split: str, start_time: int, stop_time: int, cap: int) -> tuple:
    row_slice, full_groups = row_slice_for_times(split, start_time, stop_time)
    capped_groups = np.minimum(full_groups, int(cap)).astype(np.int32)
    indices = np.empty(int(capped_groups.sum()), dtype=np.int64)
    source_offset = int(row_slice.start)
    target_offset = 0
    for full_size, capped_size in zip(full_groups, capped_groups):
        full_size = int(full_size)
        capped_size = int(capped_size)
        positions = np.linspace(0, full_size - 1, capped_size, dtype=np.int64)
        indices[target_offset:target_offset + capped_size] = source_offset + positions
        source_offset += full_size
        target_offset += capped_size
    assert source_offset == int(row_slice.stop)
    assert target_offset == indices.size == int(capped_groups.sum())
    return indices, capped_groups


def mean_cross_sectional_rank_correlation(
    left_grid: np.ndarray, right_grid: np.ndarray, groups: np.ndarray, stocks: np.ndarray
) -> float:
    values = []
    offset = 0
    for local_time, size in enumerate(groups):
        size = int(size)
        current_stocks = np.asarray(stocks[offset:offset + size], dtype=np.int32)
        values.append(rank_ic(left_grid[local_time, current_stocks], right_grid[local_time, current_stocks]))
        offset += size
    assert offset == stocks.size
    return float(np.nanmean(values))


assert abs(rank_ic(np.arange(10), np.arange(10)) - 1.0) < 1e-12
toy_groups = np.array([4, 3], dtype=np.int32)
toy_prediction = np.array([4, 1, 3, 2, 1, 3, 2], dtype=np.float32)
toy_ranked = group_rank_transform(toy_prediction, toy_groups)
assert toy_ranked.shape == toy_prediction.shape
assert np.all((toy_ranked > 0) & (toy_ranked <= 1))
print("指标、时间切片和截面秩工具自检通过。")


指标、时间切片和截面秩工具自检通过。


### 3. LightGBM 训练、分块预测和断点复用

In [4]:
LGB_PARAMS = {
    "objective": "lambdarank",
    "metric": "None",
    "learning_rate": 0.0228695,
    "num_leaves": 79,
    "min_data_in_leaf": 147,
    "feature_fraction": 0.80936,
    "bagging_fraction": 0.647764,
    "bagging_freq": 1,
    "lambda_l1": 2.35724,
    "lambda_l2": 0.238705,
    "max_bin": 127,
    "label_gain": list(range(64)),
    "lambdarank_truncation_level": 1024,
    "verbosity": -1,
    "seed": SEED,
    "feature_fraction_seed": SEED,
    "bagging_seed": SEED,
    "num_threads": NUM_THREADS,
}

ACTIVE_FOLDS = FOLDS if RUN_MODE == "full" else PREFLIGHT_FOLDS
ACTIVE_LOOKBACK = RECENT_LOOKBACK if RUN_MODE == "full" else 30

fingerprint_payload = {
    "experiment_id": EXPERIMENT_ID,
    "manifest_sha256": manifest_sha256,
    "anchor_sha256": file_sha256(ANCHOR_PATH),
    "feature_stop": FEATURE_STOP,
    "incremental_block": INCR_COLS.tolist(),
    "stock_cap": TRAIN_STOCK_CAP,
    "recent_lookback": ACTIVE_LOOKBACK,
    "base_rounds": BASE_ROUNDS,
    "rounds_candidates": list(ROUNDS_CANDIDATES),
    "weights": list(RECENT_WEIGHTS),
    "strategies": list(STRATEGIES),
    "decay_half_life": DECAY_HALF_LIFE,
    "folds": [asdict(fold) for fold in ACTIVE_FOLDS],
    "lgb_params": LGB_PARAMS,
    "seed": SEED,
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]


def build_training_arrays(
    segments: list,
    feature_set: str = "legacy",
    strategy: str = "full",
    cap: int = TRAIN_STOCK_CAP,
) -> tuple:
    prepared = []
    total_rows = 0
    for split, start_time, stop_time in segments:
        indices, groups = capped_indices_for_split(split, start_time, stop_time, cap)
        prepared.append((split, indices, groups))
        total_rows += indices.size

    feature_count = FEATURE_STOP if feature_set == "legacy" else ENHANCED_FEATURE_COUNT
    X_train = np.empty((total_rows, feature_count), dtype=np.float32)
    y_train = np.empty(total_rows, dtype=np.int8)
    row_time = np.empty(total_rows, dtype=np.int64)
    all_groups = []
    offset = 0
    for split, indices, groups in prepared:
        stop = offset + indices.size
        X_train[offset:stop, :FEATURE_STOP] = tree[split][indices, :FEATURE_STOP]
        if feature_set != "legacy":
            X_train[offset:stop, FEATURE_STOP:] = bank_incremental(split, indices)
        y_train[offset:stop] = common[split]["relevance"][indices]
        row_time[offset:stop] = common[split]["time"][indices]
        all_groups.append(groups)
        offset = stop
    assert offset == total_rows
    final_groups = np.concatenate(all_groups).astype(np.int32)
    assert int(final_groups.sum()) == total_rows

    sample_weights = None
    if strategy == "decay_1200":
        horizon = segments[-1][2]
        age = (horizon - 1 - row_time).astype(np.float64)
        sample_weights = np.power(0.5, age / DECAY_HALF_LIFE)
        sample_weights = (sample_weights / sample_weights.mean()).astype(np.float32)
    return X_train, y_train, final_groups, sample_weights


def train_ranker(
    segments: list,
    boost_rounds: int,
    feature_set: str = "legacy",
    strategy: str = "full",
):
    import lightgbm as lgb

    X_train, y_train, train_groups, sample_weights = build_training_arrays(
        segments, feature_set, strategy
    )
    dataset = lgb.Dataset(
        X_train,
        label=y_train,
        group=train_groups,
        weight=sample_weights,
        free_raw_data=True,
    )
    started_at = time.time()
    model = lgb.train(
        LGB_PARAMS,
        dataset,
        num_boost_round=int(boost_rounds),
        callbacks=[lgb.log_evaluation(0)],
    )
    elapsed = time.time() - started_at
    train_rows = int(X_train.shape[0])
    del dataset, X_train, y_train, train_groups, sample_weights
    gc.collect()
    return model, {"training_seconds": elapsed, "train_rows": train_rows}


def predict_interval(
    model,
    split: str,
    start_time: int,
    stop_time: int,
    num_iteration: int,
    feature_set: str = "legacy",
    chunk_size: int = 200_000,
) -> tuple:
    rows, groups = row_slice_for_times(split, start_time, stop_time)
    prediction = np.empty(int(rows.stop) - int(rows.start), dtype=np.float32)
    feature_count = FEATURE_STOP if feature_set == "legacy" else ENHANCED_FEATURE_COUNT
    for begin in range(int(rows.start), int(rows.stop), chunk_size):
        end = min(begin + chunk_size, int(rows.stop))
        block = tree[split][begin:end, :FEATURE_STOP]
        if feature_set != "legacy":
            block = np.column_stack([
                block,
                bank_incremental(split, np.arange(begin, end)),
            ])
        assert block.shape[1] == feature_count
        prediction[begin - int(rows.start):end - int(rows.start)] = model.predict(
            block, num_iteration=int(num_iteration)
        ).astype(np.float32)
    return prediction, groups


def interval_target(split: str, start_time: int, stop_time: int) -> np.ndarray:
    rows, _ = row_slice_for_times(split, start_time, stop_time)
    return np.asarray(common[split]["y"][rows], dtype=np.float32)


def save_model_text_atomic(model, path: Path) -> None:
    atomic_write_text(path, model.model_to_string())


print("模型参数指纹：", RUN_FINGERPRINT)


模型参数指纹： ecd185fd574392f3


In [5]:
if RUN_MODE == "preflight":
    smoke_model, smoke_train_info = train_ranker(
        [("train", 486, 490)], boost_rounds=2, feature_set="legacy", strategy="full"
    )
    smoke_prediction, smoke_groups = predict_interval(
        smoke_model, "train", 490, 492, num_iteration=2, feature_set="legacy"
    )
    smoke_target = interval_target("train", 490, 492)
    assert smoke_prediction.size == smoke_target.size == int(smoke_groups.sum())
    assert np.isfinite(smoke_prediction).all()

    smoke_enhanced, _ = train_ranker(
        [("train", 486, 490)], boost_rounds=2, feature_set="enhanced", strategy="full"
    )
    smoke_enhanced_pred, _ = predict_interval(
        smoke_enhanced, "train", 490, 492, num_iteration=2, feature_set="enhanced"
    )
    assert smoke_enhanced_pred.size == smoke_prediction.size
    assert np.isfinite(smoke_enhanced_pred).all()
    del smoke_enhanced
    gc.collect()

    Xd, yd, gd, wd = build_training_arrays(
        [("train", 486, 490)], feature_set="legacy", strategy="decay_1200"
    )
    assert wd is not None and np.isclose(float(wd.mean()), 1.0) and np.all(wd > 0)
    assert np.isfinite(Xd).all()

    del smoke_model
    gc.collect()
    print("Preflight：legacy / enhanced / decay_1200 冒烟测试全部通过。")
else:
    print("完整模式：跳过重复的轻量冒烟测试，直接进入走步训练。")


完整模式：跳过重复的轻量冒烟测试，直接进入走步训练。


## Results

### 4. Step A：exp_009 基线复现（3 折走步）

每折训练：完整历史锚点（legacy_328、8 轮）+ 4 个专家变体（legacy/enhanced × full/decay_1200，各最多 32 轮，预测 8/16/32 轮）。预测写入带配置指纹的断点缓存，重复运行直接复用。

In [6]:
fold_rows = []
fold_data = {}

for fold in ACTIVE_FOLDS:
    valid_target = interval_target("train", fold.valid_start, fold.valid_stop)
    _, valid_groups = row_slice_for_times("train", fold.valid_start, fold.valid_stop)
    recent_start = max(fold.train_start, fold.train_stop - ACTIVE_LOOKBACK)
    recent_segments = [("train", recent_start, fold.train_stop)]

    anchor_cache = RUNTIME_CACHE_DIR / f"{fold.name}_{RUN_FINGERPRINT}_anchor.npz"
    if anchor_cache.exists():
        with np.load(anchor_cache) as cached:
            anchor_prediction = np.asarray(cached["anchor_prediction"], dtype=np.float32)
        print(f"复用 {fold.name} 锚点预测缓存。", flush=True)
    else:
        print(f"{fold.name}：训练完整历史锚点 [486, {fold.train_stop})。", flush=True)
        base_model, base_train_info = train_ranker(
            [("train", fold.train_start, fold.train_stop)], BASE_ROUNDS, "legacy", "full"
        )
        anchor_prediction, _ = predict_interval(
            base_model, "train", fold.valid_start, fold.valid_stop, BASE_ROUNDS, "legacy"
        )
        del base_model
        gc.collect()
        atomic_save_npz(anchor_cache, anchor_prediction=anchor_prediction)
    assert anchor_prediction.size == valid_target.size

    anchor_rank = group_rank_transform(anchor_prediction, valid_groups)
    anchor_metrics = score_prediction(anchor_rank, valid_target, valid_groups)

    experts = {}
    expert_ic = {}
    for feature_set in ("legacy", "enhanced"):
        for strategy in STRATEGIES:
            cache_path = RUNTIME_CACHE_DIR / (
                f"{fold.name}_{RUN_FINGERPRINT}_expert_{feature_set}_{strategy}.npz"
            )
            if cache_path.exists():
                with np.load(cache_path) as cached:
                    predictions = {
                        r: np.asarray(cached[f"p{r}"], dtype=np.float32)
                        for r in ROUNDS_CANDIDATES
                    }
                print(f"复用 {fold.name} 专家缓存 {feature_set}/{strategy}。", flush=True)
            else:
                print(
                    f"{fold.name}：训练近期专家 [{recent_start}, {fold.train_stop}) "
                    f"{feature_set}/{strategy}（{MAX_EXPERT_ROUNDS} 轮）。",
                    flush=True,
                )
                recent_model, recent_train_info = train_ranker(
                    recent_segments, MAX_EXPERT_ROUNDS, feature_set, strategy
                )
                predictions = {}
                for r in ROUNDS_CANDIDATES:
                    p, _ = predict_interval(
                        recent_model, "train", fold.valid_start, fold.valid_stop, r, feature_set
                    )
                    predictions[r] = p
                del recent_model
                gc.collect()
                atomic_save_npz(
                    cache_path,
                    **{f"p{r}": predictions[r] for r in ROUNDS_CANDIDATES},
                )
            for r in ROUNDS_CANDIDATES:
                assert predictions[r].size == valid_target.size
                experts[(feature_set, strategy, r)] = group_rank_transform(predictions[r], valid_groups)
                expert_ic[(feature_set, strategy, r)] = score_prediction(
                    experts[(feature_set, strategy, r)], valid_target, valid_groups
                )

    fold_data[fold.name] = {
        "fold": fold,
        "valid_target": valid_target,
        "valid_groups": valid_groups,
        "anchor_rank": anchor_rank,
        "anchor_metrics": anchor_metrics,
        "expert_rank": experts,
        "expert_ic": expert_ic,
    }

    fold_rows.append({
        "fold": fold.name,
        "feature_set": "legacy",
        "strategy": "full",
        "expert_rounds": 0,
        **anchor_metrics,
        "mean_delta": 0.0,
        "late_delta": 0.0,
    })
    for (feature_set, strategy, r), rank in experts.items():
        metrics = expert_ic[(feature_set, strategy, r)]
        fold_rows.append({
            "fold": fold.name,
            "feature_set": feature_set,
            "strategy": strategy,
            "expert_rounds": r,
            **metrics,
            "mean_delta": metrics["mean_rank_ic"] - anchor_metrics["mean_rank_ic"],
            "late_delta": metrics["late_half_rank_ic"] - anchor_metrics["late_half_rank_ic"],
        })

fold_results = pd.DataFrame(fold_rows)
atomic_write_csv(RUN_DIR / "fold_results.csv", fold_results)
display(fold_results[fold_results["fold"] == fold.name].sort_values(
    ["feature_set", "strategy", "expert_rounds"]
).head(12))
print(f"走步结果行数：{len(fold_results)}")


fold_1：训练完整历史锚点 [486, 2189)。
fold_1：训练近期专家 [487, 2189) legacy/full（32 轮）。
fold_1：训练近期专家 [487, 2189) legacy/decay_1200（32 轮）。
fold_1：训练近期专家 [487, 2189) enhanced/full（32 轮）。
fold_1：训练近期专家 [487, 2189) enhanced/decay_1200（32 轮）。
fold_2：训练完整历史锚点 [486, 2432)。
fold_2：训练近期专家 [730, 2432) legacy/full（32 轮）。
fold_2：训练近期专家 [730, 2432) legacy/decay_1200（32 轮）。
fold_2：训练近期专家 [730, 2432) enhanced/full（32 轮）。
fold_2：训练近期专家 [730, 2432) enhanced/decay_1200（32 轮）。
fold_3：训练完整历史锚点 [486, 2675)。
fold_3：训练近期专家 [973, 2675) legacy/full（32 轮）。
fold_3：训练近期专家 [973, 2675) legacy/decay_1200（32 轮）。
fold_3：训练近期专家 [973, 2675) enhanced/full（32 轮）。
fold_3：训练近期专家 [973, 2675) enhanced/decay_1200（32 轮）。


,fold,feature_set,strategy,expert_rounds,mean_rank_ic,std_rank_ic,late_half_rank_ic,worst_quarter_rank_ic,negative_time_share,mean_delta,late_delta
36,fold_3,enhanced,decay_1200,8,0.088709,0.120110,0.085034,0.059596,0.213992,0.004937,0.003085
37,fold_3,enhanced,decay_1200,16,0.089889,0.121480,0.085928,0.060338,0.213992,0.006117,0.003980
38,fold_3,enhanced,decay_1200,32,0.093705,0.121723,0.090457,0.064784,0.201646,0.009933,0.008508
33,fold_3,enhanced,full,8,0.084938,0.122947,0.083790,0.057953,0.251029,0.001165,0.001842
34,fold_3,enhanced,full,16,0.086177,0.123527,0.084496,0.057436,0.251029,0.002405,0.002548
35,fold_3,enhanced,full,32,0.089196,0.124706,0.087656,0.059154,0.222222,0.005424,0.005708
30,fold_3,legacy,decay_1200,8,0.086409,0.127943,0.084086,0.057298,0.255144,0.002637,0.002138
31,fold_3,legacy,decay_1200,16,0.088196,0.127236,0.085409,0.059001,0.242798,0.004424,0.003461
32,fold_3,legacy,decay_1200,32,0.090681,0.125558,0.087422,0.060359,0.222222,0.006909,0.005474
26,fold_3,legacy,full,0,0.083772,0.126614,0.081948,0.054333,0.259259,0.000000,0.000000


走步结果行数：39


### 5. Step A 验证：与 exp_009 折内结果核对

若环境与数据契约可复现 exp_009，则 (16, 0.25) 融合与 8 轮锚点在三个折内的平均 RankIC 应与 `exp_009/fold_results.csv` 高度一致（容差 5e-4）。

In [7]:
if RUN_MODE == "full":
    repro_rows = []
    for fold_name, targets in EXP009_FOLD_TARGETS.items():
        fd = fold_data[fold_name]
        anchor_ic = fd["anchor_metrics"]["mean_rank_ic"]
        blend_rank = (0.75 * fd["anchor_rank"]) + (0.25 * fd["expert_rank"][("legacy", "full", 16)])
        blend_ic = score_prediction(blend_rank, fd["valid_target"], fd["valid_groups"])["mean_rank_ic"]
        repro_rows.append({
            "fold": fold_name,
            "anchor_target": targets["anchor"],
            "anchor_repro": anchor_ic,
            "anchor_diff": anchor_ic - targets["anchor"],
            "blend16_25_target": targets["blend16_25"],
            "blend16_25_repro": blend_ic,
            "blend_diff": blend_ic - targets["blend16_25"],
        })
    repro_df = pd.DataFrame(repro_rows)
    atomic_write_csv(RUN_DIR / "exp009_reproduction.csv", repro_df)
    display(repro_df)
    max_diff = float(max(repro_df["anchor_diff"].abs().max(), repro_df["blend_diff"].abs().max()))
    exp009_reproduced = bool(max_diff <= REPRODUCTION_TOLERANCE)
    print(pd.Series({
        "exp009_reproduced": exp009_reproduced,
        "max_abs_diff": max_diff,
        "tolerance": REPRODUCTION_TOLERANCE,
    }))
else:
    exp009_reproduced = True
    print("Preflight：跳过 exp_009 复现核对。")


,fold,anchor_target,anchor_repro,anchor_diff,blend16_25_target,blend16_25_repro,blend_diff
0,fold_1,0.112031,0.112031,0.0,0.113018,0.113018,0.0
1,fold_2,0.097117,0.097117,0.0,0.098006,0.098006,0.0
2,fold_3,0.083772,0.083772,0.0,0.085069,0.085069,0.0


exp009_reproduced      True
max_abs_diff            0.0
tolerance            0.0005
dtype: object


### 6. Step B：特征诊断（legacy_328 vs 增强 433）

比较纯专家（不融合）在 16 轮的折内平均 RankIC。保留条件：**3 折均为正 且 均值增量 ≥ 0.0005**，否则回退 `legacy_328`。

In [8]:
diag_rows = []
for fold_name, fd in fold_data.items():
    legacy_ic = fd["expert_ic"][("legacy", "full", 16)]["mean_rank_ic"]
    enhanced_ic = fd["expert_ic"][("enhanced", "full", 16)]["mean_rank_ic"]
    diag_rows.append({
        "fold": fold_name,
        "legacy_ic": legacy_ic,
        "enhanced_ic": enhanced_ic,
        "delta": enhanced_ic - legacy_ic,
    })

diag_df = pd.DataFrame(diag_rows)
diag_mean_gain = float(diag_df["delta"].mean())
diag_all_positive = bool((diag_df["delta"] > 0).all())
feature_enhanced_kept = bool(
    RUN_MODE == "full" and diag_all_positive and diag_mean_gain >= DIAGNOSIS_MIN_MEAN_GAIN
)
if RUN_MODE != "full":
    feature_enhanced_kept = False
    print("Preflight：增强特征保留判定关闭，回退 legacy_328。")
SELECTED_FEATURE_SET = "enhanced" if feature_enhanced_kept else "legacy"
atomic_write_csv(RUN_DIR / "feature_diagnosis.csv", diag_df)
display(diag_df)
print(pd.Series({
    "diagnosis_mean_gain": diag_mean_gain,
    "all_three_positive": diag_all_positive,
    "required_mean_gain": DIAGNOSIS_MIN_MEAN_GAIN,
    "feature_enhanced_kept": feature_enhanced_kept,
    "selected_feature_set": SELECTED_FEATURE_SET,
}))


,fold,legacy_ic,enhanced_ic,delta
0,fold_1,0.110115,0.110803,0.000688
1,fold_2,0.096467,0.098068,0.001600
2,fold_3,0.085794,0.086177,0.000383


diagnosis_mean_gain       0.00089
all_three_positive           True
required_mean_gain         0.0005
feature_enhanced_kept        True
selected_feature_set     enhanced
dtype: object


### 7. Step C：时间策略（full vs decay_1200）

在已选特征集上比较近期窗口内的均匀权重（full）与指数时间衰减（decay_1200，半衰期 1200）。选择条件：**3 折均值与后段均不劣于 full**。

In [9]:
strat_rows = []
for fold_name, fd in fold_data.items():
    full_metrics = fd["expert_ic"][(SELECTED_FEATURE_SET, "full", 16)]
    decay_metrics = fd["expert_ic"][(SELECTED_FEATURE_SET, "decay_1200", 16)]
    strat_rows.append({
        "fold": fold_name,
        "full_mean_ic": full_metrics["mean_rank_ic"],
        "decay_mean_ic": decay_metrics["mean_rank_ic"],
        "mean_delta": decay_metrics["mean_rank_ic"] - full_metrics["mean_rank_ic"],
        "full_late_ic": full_metrics["late_half_rank_ic"],
        "decay_late_ic": decay_metrics["late_half_rank_ic"],
        "late_delta": decay_metrics["late_half_rank_ic"] - full_metrics["late_half_rank_ic"],
    })

strat_df = pd.DataFrame(strat_rows)
strat_mean_delta = float(strat_df["mean_delta"].mean())
strat_late_delta = float(strat_df["late_delta"].mean())
decay_selected = bool(
    RUN_MODE == "full" and strat_mean_delta >= 0.0 and strat_late_delta >= 0.0
)
if RUN_MODE != "full":
    decay_selected = False
    print("Preflight：时间策略判定关闭，使用 full。")
SELECTED_STRATEGY = "decay_1200" if decay_selected else "full"
atomic_write_csv(RUN_DIR / "time_strategy.csv", strat_df)
display(strat_df)
print(pd.Series({
    "decay_vs_full_mean_delta": strat_mean_delta,
    "decay_vs_full_late_delta": strat_late_delta,
    "decay_selected": decay_selected,
    "selected_strategy": SELECTED_STRATEGY,
}))


,fold,full_mean_ic,decay_mean_ic,mean_delta,full_late_ic,decay_late_ic,late_delta
0,fold_1,0.110803,0.113484,0.002681,0.132370,0.133316,0.000946
1,fold_2,0.098068,0.095043,-0.003025,0.066304,0.065668,-0.000636
2,fold_3,0.086177,0.089889,0.003712,0.084496,0.085928,0.001431


decay_vs_full_mean_delta      0.001123
decay_vs_full_late_delta      0.000581
decay_selected                    True
selected_strategy           decay_1200
dtype: object


### 8. Step D：轮数与权重选择（含 exp_004 稳定性平台）

在已选特征集与时间策略上，对所有 (轮数, 权重) 组合计算融合折内指标；`plateau_span` 为 ±0.1 邻近权重在全窗口的 RankIC 最大跨度，须 < 0.001。选择规则沿用 exp_009：在 `qualified` 内取最佳平均增益 95% 范围内的最低权重。

In [10]:
weight_rows = []
for fold_name, fd in fold_data.items():
    anchor_ic = fd["anchor_metrics"]["mean_rank_ic"]
    anchor_late = fd["anchor_metrics"]["late_half_rank_ic"]
    for r in ROUNDS_CANDIDATES:
        expert_rank = fd["expert_rank"][(SELECTED_FEATURE_SET, SELECTED_STRATEGY, r)]
        for w in RECENT_WEIGHTS[1:]:
            blended = (1.0 - w) * fd["anchor_rank"] + w * expert_rank
            metrics = score_prediction(blended, fd["valid_target"], fd["valid_groups"])
            weight_rows.append({
                "fold": fold_name,
                "expert_rounds": r,
                "recent_weight": w,
                **metrics,
                "mean_delta": metrics["mean_rank_ic"] - anchor_ic,
                "late_delta": metrics["late_half_rank_ic"] - anchor_late,
            })

fold_weight_frame = pd.DataFrame(weight_rows)
atomic_write_csv(RUN_DIR / "blend_fold_results.csv", fold_weight_frame)

summary_rows = []
for (r, w), frame in fold_weight_frame.groupby(["expert_rounds", "recent_weight"]):
    summary_rows.append({
        "expert_rounds": int(r),
        "recent_weight": float(w),
        "mean_rank_ic": float(frame["mean_rank_ic"].mean()),
        "fold_std": float(frame["mean_rank_ic"].std(ddof=0)),
        "late_half_rank_ic": float(frame["late_half_rank_ic"].mean()),
        "worst_quarter_rank_ic": float(frame["worst_quarter_rank_ic"].min()),
        "mean_improvement": float(frame["mean_delta"].mean()),
        "worst_fold_delta": float(frame["mean_delta"].min()),
        "late_mean_delta": float(frame["late_delta"].mean()),
        "positive_fold_count": int((frame["mean_delta"] > 0).sum()),
    })
weight_search = pd.DataFrame(summary_rows)


def plateau_span_for(rounds: int, weight: float) -> float:
    grid = RECENT_WEIGHTS
    w_lo = min(grid, key=lambda g: abs(g - max(0.0, weight - PLATEAU_WINDOW)))
    w_hi = min(grid, key=lambda g: abs(g - min(float(grid[-1]), weight + PLATEAU_WINDOW)))
    spans = []
    for fold_name, fd in fold_data.items():
        ics = []
        for cand_w in (w_lo, weight, w_hi):
            blended = (1.0 - cand_w) * fd["anchor_rank"] + cand_w * fd["expert_rank"][
                (SELECTED_FEATURE_SET, SELECTED_STRATEGY, rounds)
            ]
            ics.append(score_prediction(blended, fd["valid_target"], fd["valid_groups"])["mean_rank_ic"])
        spans.append(max(ics) - min(ics))
    return float(max(spans))


weight_search["plateau_span"] = weight_search.apply(
    lambda row: plateau_span_for(int(row["expert_rounds"]), float(row["recent_weight"])),
    axis=1,
)

weight_search["qualified"] = (
    (weight_search["recent_weight"] > 0)
    & (weight_search["recent_weight"] <= 0.25)
    & (weight_search["positive_fold_count"] >= MIN_POSITIVE_FOLDS)
    & (weight_search["mean_improvement"] >= MIN_MEAN_IMPROVEMENT)
    & (weight_search["worst_fold_delta"] >= -MAX_WORST_FOLD_DROP)
    & (weight_search["late_mean_delta"] >= -MAX_LATE_MEAN_DROP)
    & (weight_search["plateau_span"] < PLATEAU_MAX_SPAN)
)

qualified = weight_search[weight_search["qualified"]].copy()
if qualified.empty:
    selected_expert_rounds = 8
    selected_recent_weight = 0.0
    selection_reason = "没有候选通过 Train 内稳定性门槛，回退到正式锚点。"
else:
    best_improvement = float(qualified["mean_improvement"].max())
    near_best = qualified[
        qualified["mean_improvement"] >= best_improvement * 0.95
    ].sort_values(
        ["recent_weight", "fold_std", "expert_rounds"],
        ascending=[True, True, True],
    )
    selected = near_best.iloc[0]
    selected_expert_rounds = int(selected["expert_rounds"])
    selected_recent_weight = float(selected["recent_weight"])
    selection_reason = "选择达到最佳平均增益 95% 范围内的最低近期权重。"

if selected_recent_weight == 0.0:
    weight_search["selected"] = (
        (weight_search["expert_rounds"] == 8) & np.isclose(weight_search["recent_weight"], 0.0)
    )
else:
    weight_search["selected"] = (
        (weight_search["expert_rounds"] == selected_expert_rounds)
        & np.isclose(weight_search["recent_weight"], selected_recent_weight)
    )
atomic_write_csv(RUN_DIR / "weight_search.csv", weight_search)

display(weight_search.sort_values(["qualified", "mean_improvement"], ascending=[False, False]).head(16))
print(pd.Series({
    "selected_expert_rounds": selected_expert_rounds,
    "selected_recent_weight": selected_recent_weight,
    "selected_feature_set": SELECTED_FEATURE_SET,
    "selected_strategy": SELECTED_STRATEGY,
    "selection_reason": selection_reason,
}))


,expert_rounds,recent_weight,mean_rank_ic,fold_std,late_half_rank_ic,worst_quarter_rank_ic,mean_improvement,worst_fold_delta,late_mean_delta,positive_fold_count,plateau_span,qualified,selected
20,32,0.35,0.101030,0.011563,0.097546,0.058451,0.003390,0.001489,0.002913,3,0.001176,False,False
19,32,0.30,0.100642,0.011588,0.097239,0.057901,0.003002,0.001351,0.002605,3,0.001792,False,False
18,32,0.25,0.100213,0.011600,0.096883,0.057332,0.002573,0.001182,0.002250,3,0.002429,False,False
13,16,0.35,0.099900,0.011491,0.096469,0.057035,0.002259,0.000521,0.001836,3,0.000817,False,False
17,32,0.20,0.099764,0.011604,0.096507,0.056778,0.002123,0.001003,0.001874,3,0.002504,False,False
6,8,0.35,0.099719,0.011349,0.096269,0.056869,0.002078,0.000748,0.001636,3,0.000732,False,False
12,16,0.30,0.099681,0.011516,0.096321,0.056693,0.002040,0.000534,0.001688,3,0.001259,False,False
5,8,0.30,0.099538,0.011412,0.096163,0.056541,0.001897,0.000735,0.001530,3,0.001140,False,False
11,16,0.25,0.099418,0.011532,0.096125,0.056341,0.001778,0.000507,0.001492,3,0.001736,False,False
4,8,0.25,0.099318,0.011457,0.096010,0.056209,0.001678,0.000684,0.001377,3,0.001574,False,False


selected_expert_rounds                               8
selected_recent_weight                             0.0
selected_feature_set                          enhanced
selected_strategy                           decay_1200
selection_reason          没有候选通过 Train 内稳定性门槛，回退到正式锚点。
dtype: object


### 9. 官方 Valid 一次性晋级检查

选定特征集/策略/轮数/权重后，使用完整 Train 重训锚点与近期专家，在官方 Valid 上只检查一次。该检查只决定 `promoted` 状态，不搜索或修改任何权重。

In [11]:
def prediction_vector_to_grid(
    prediction: np.ndarray,
    split: str,
    time_start: int,
    time_points: int,
) -> np.ndarray:
    grid = np.full((time_points, STOCK_COUNT), 0.5, dtype=np.float32)
    times = np.asarray(common[split]["time"], dtype=np.int32)
    stocks = np.asarray(common[split]["stock"], dtype=np.int32)
    grid[times - time_start, stocks] = prediction
    return grid


if RUN_MODE == "preflight":
    official_base_metrics = {
        "mean_rank_ic": ANCHOR_EXPECTED_VALID_IC,
        "std_rank_ic": 0.10,
        "late_half_rank_ic": 0.0918,
        "worst_quarter_rank_ic": 0.0600,
        "negative_time_share": 0.20,
    }
    official_expert_metrics = {**official_base_metrics, "mean_rank_ic": 0.0940}
    official_blend_metrics = {
        **official_base_metrics,
        "mean_rank_ic": ANCHOR_EXPECTED_VALID_IC + 0.0006,
        "late_half_rank_ic": 0.0922,
        "worst_quarter_rank_ic": 0.0602,
    }
    official_anchor_reproduced = True
    valid_grid = np.full((VALID_STOP - VALID_START, STOCK_COUNT), 0.5, dtype=np.float32)
else:
    official_cache_path = RUNTIME_CACHE_DIR / (
        f"official_valid_{RUN_FINGERPRINT}_{SELECTED_FEATURE_SET}_{SELECTED_STRATEGY}"
        f"_r{selected_expert_rounds}.npz"
    )
    valid_target = np.asarray(common["valid"]["y"], dtype=np.float32)
    valid_groups = np.asarray(common["valid"]["groups"], dtype=np.int32)

    if official_cache_path.exists():
        with np.load(official_cache_path) as cached:
            official_base_prediction = np.asarray(cached["base_prediction"], dtype=np.float32)
            official_expert_prediction = np.asarray(cached["expert_prediction"], dtype=np.float32)
        print("复用官方 Valid 预测缓存。")
    else:
        print("训练官方 Valid 锚点模型（legacy_328 / full / 8 轮）。", flush=True)
        official_base_model, _ = train_ranker(
            [("train", TRAIN_START, TRAIN_STOP)], BASE_ROUNDS, "legacy", "full"
        )
        official_base_prediction, _ = predict_interval(
            official_base_model, "valid", VALID_START, VALID_STOP, BASE_ROUNDS, "legacy"
        )
        del official_base_model
        gc.collect()

        official_recent_start = TRAIN_STOP - RECENT_LOOKBACK
        print(
            f"训练官方 Valid 近期专家 [{official_recent_start}, {TRAIN_STOP}) "
            f"{SELECTED_FEATURE_SET}/{SELECTED_STRATEGY}/r{selected_expert_rounds}。",
            flush=True,
        )
        official_expert_model, _ = train_ranker(
            [("train", official_recent_start, TRAIN_STOP)],
            selected_expert_rounds,
            SELECTED_FEATURE_SET,
            SELECTED_STRATEGY,
        )
        official_expert_prediction, _ = predict_interval(
            official_expert_model,
            "valid",
            VALID_START,
            VALID_STOP,
            selected_expert_rounds,
            SELECTED_FEATURE_SET,
        )
        del official_expert_model
        gc.collect()
        atomic_save_npz(
            official_cache_path,
            base_prediction=official_base_prediction,
            expert_prediction=official_expert_prediction,
        )

    assert official_base_prediction.size == official_expert_prediction.size == valid_target.size
    official_base_rank = group_rank_transform(official_base_prediction, valid_groups)
    official_expert_rank = group_rank_transform(official_expert_prediction, valid_groups)
    official_blend_rank = (
        (1.0 - selected_recent_weight) * official_base_rank
        + selected_recent_weight * official_expert_rank
    )
    official_base_metrics = score_prediction(official_base_rank, valid_target, valid_groups)
    official_expert_metrics = score_prediction(official_expert_rank, valid_target, valid_groups)
    official_blend_metrics = score_prediction(official_blend_rank, valid_target, valid_groups)
    official_anchor_reproduced = (
        abs(official_base_metrics["mean_rank_ic"] - ANCHOR_EXPECTED_VALID_IC)
        <= ANCHOR_REPRODUCTION_TOLERANCE
    )
    valid_grid = prediction_vector_to_grid(
        official_blend_rank, "valid", VALID_START, VALID_STOP - VALID_START
    )

official_mean_delta = official_blend_metrics["mean_rank_ic"] - official_base_metrics["mean_rank_ic"]
official_late_delta = official_blend_metrics["late_half_rank_ic"] - official_base_metrics["late_half_rank_ic"]
official_worst_delta = official_blend_metrics["worst_quarter_rank_ic"] - official_base_metrics["worst_quarter_rank_ic"]

if selected_recent_weight > 0.0:
    selected_plateau_span = plateau_span_for(selected_expert_rounds, selected_recent_weight)
else:
    selected_plateau_span = 0.0
plateau_passed = bool(selected_plateau_span < PLATEAU_MAX_SPAN)

promoted = bool(
    RUN_MODE == "full"
    and selected_recent_weight > 0
    and official_anchor_reproduced
    and official_mean_delta >= OFFICIAL_MIN_IMPROVEMENT
    and official_late_delta >= -OFFICIAL_MAX_LATE_DROP
    and official_worst_delta >= -OFFICIAL_MAX_WORST_QUARTER_DROP
    and plateau_passed
)

official_valid_results = pd.DataFrame([
    {"model": "anchor", **official_base_metrics},
    {"model": "recent_expert", **official_expert_metrics},
    {"model": "selected_blend", **official_blend_metrics},
])
atomic_write_csv(RUN_DIR / "official_valid_results.csv", official_valid_results)
atomic_save_npy(RUN_DIR / "valid_prediction.npy", valid_grid)
display(official_valid_results)
print(pd.Series({
    "anchor_reproduced": official_anchor_reproduced,
    "official_mean_delta": official_mean_delta,
    "official_late_delta": official_late_delta,
    "official_worst_quarter_delta": official_worst_delta,
    "selected_plateau_span": selected_plateau_span,
    "plateau_passed": plateau_passed,
    "promoted": promoted,
}))


训练官方 Valid 锚点模型（legacy_328 / full / 8 轮）。
训练官方 Valid 近期专家 [1216, 2918) enhanced/decay_1200/r8。


,model,mean_rank_ic,std_rank_ic,late_half_rank_ic,worst_quarter_rank_ic,negative_time_share
0,anchor,0.092940,0.109053,0.091868,0.059262,0.197531
1,recent_expert,0.081221,0.101758,0.076863,0.057405,0.213992
2,selected_blend,0.092940,0.109053,0.091868,0.059262,0.197531


anchor_reproduced                True
official_mean_delta               0.0
official_late_delta               0.0
official_worst_quarter_delta      0.0
selected_plateau_span             0.0
plateau_passed                   True
promoted                        False
dtype: object


### 10. 可选 TCN 低权重差异信号（默认关闭）

复刻 exp_002 配置（8 残差块、dilation 1..128、kernel 3、GroupNorm 4、dropout 0.1、AdamW lr=1e-3、8 epoch）。仅在 `DSCR_EXP010_USE_TCN=1` 时训练：每折近期窗口训练一个 TCN 并对折内验证预测，随后在 {0.05, 0.10, 0.15} 网格上选择 TCN 权重（作用于已选锚点+专家融合），最后训练全量 TCN 预测 Test。默认关闭时不改变任何结果。

In [12]:
tcn_weight = 0.0
tcn_grid = None
if RUN_MODE == "full" and USE_TCN == 1:
    import torch
    import torch.nn as nn

    WINDOW_SIZE = 486
    TCN_SEED = SEED
    TCN_FEATURE_COUNT = 40
    MODEL_INPUT_CHANNELS = TCN_FEATURE_COUNT + 1
    HIDDEN_CHANNELS = 32
    DILATIONS = (1, 2, 4, 8, 16, 32, 64, 128)
    DROPOUT = 0.1
    TCN_BATCH = 128
    MAX_EPOCHS = 8
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    TRAIN_TIME_BINS = 64
    TIMES_PER_BIN = 8
    STOCKS_PER_TIME = 200

    class CausalResidualBlock(nn.Module):
        def __init__(self, channels, dilation, dropout):
            super().__init__()
            self.left_padding = nn.ConstantPad1d((2 * dilation, 0), 0.0)
            self.convolution = nn.Conv1d(channels, channels, kernel_size=3, dilation=dilation)
            self.normalization = nn.GroupNorm(4, channels)
            self.activation = nn.GELU()
            self.dropout = nn.Dropout(dropout)

        def forward(self, inputs):
            hidden = self.left_padding(inputs)
            hidden = self.convolution(hidden)
            hidden = self.normalization(hidden)
            hidden = self.activation(hidden)
            hidden = self.dropout(hidden)
            return inputs + hidden

    class WindowTCN(nn.Module):
        def __init__(self, input_channels, hidden_channels, dilations, dropout):
            super().__init__()
            self.projection = nn.Conv1d(input_channels, hidden_channels, kernel_size=1)
            self.projection_activation = nn.GELU()
            self.blocks = nn.ModuleList([
                CausalResidualBlock(hidden_channels, d, dropout) for d in dilations
            ])
            self.head = nn.Sequential(
                nn.Linear(hidden_channels + 1, hidden_channels),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_channels, 1),
                nn.Sigmoid(),
            )

        def forward(self, sequence_inputs, coverage):
            hidden = self.projection_activation(self.projection(sequence_inputs))
            for block in self.blocks:
                hidden = block(hidden)
            final_hidden = hidden[:, :, -1]
            combined = torch.cat([final_hidden, coverage], dim=1)
            return self.head(combined)

    torch.manual_seed(TCN_SEED)
    np.random.seed(TCN_SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    AMP_ENABLED = DEVICE.type == "cuda"

    SEQ_X = np.load(DATASET_DIR / "sequence" / "X.npy", mmap_mode="r")
    MASK_X = np.load(DATASET_DIR / "sequence" / "mask_x.npy", mmap_mode="r")
    assert SEQ_X.shape == (3603, 5282, TCN_FEATURE_COUNT)

    feature_sum = np.zeros(TCN_FEATURE_COUNT, dtype=np.float64)
    feature_sq_sum = np.zeros(TCN_FEATURE_COUNT, dtype=np.float64)
    sample_count = 0
    for chunk_start in range(TRAIN_START, TRAIN_STOP, 16):
        chunk_stop = min(chunk_start + 16, TRAIN_STOP)
        block = SEQ_X[chunk_start:chunk_stop]
        valid_block = MASK_X[chunk_start:chunk_stop]
        valid_features = block[valid_block]
        feature_sum += valid_features.sum(axis=0, dtype=np.float64)
        feature_sq_sum += np.square(valid_features, dtype=np.float64).sum(axis=0)
        sample_count += valid_features.shape[0]
    feature_mean = (feature_sum / sample_count).astype(np.float32)
    feature_var = np.maximum(feature_sq_sum / sample_count - np.square(feature_mean), 1e-6)
    feature_std = np.sqrt(feature_var).astype(np.float32)

    # mask_y 与 y1 需要整面板；sequence 目录未缓存，从 data.z 完整读取
    import pickle
    import zstandard as zstd

    compressed = pd.read_pickle(PROJECT_ROOT / "data.z")
    decompressed = zstd.ZstdDecompressor().decompress(compressed)
    full_data = pickle.loads(decompressed)
    MASK_Y_TCN = full_data["mask_y"]
    Y1_TCN = full_data["y1"]
    del full_data, decompressed, compressed
    gc.collect()

    def labeled_stock_indices(time_idx):
        return np.flatnonzero(MASK_Y_TCN[time_idx] & np.isfinite(Y1_TCN[time_idx]))

    tcn_rng = np.random.default_rng(TCN_SEED)

    def build_model_batch(time_idx, stock_indices):
        stock_indices = np.asarray(stock_indices, dtype=np.int64)
        window_start = time_idx - WINDOW_SIZE + 1
        feature_window = SEQ_X[window_start:time_idx + 1, stock_indices, :]
        valid_window = MASK_X[window_start:time_idx + 1, stock_indices]
        valid_counts = valid_window.sum(axis=0, dtype=np.int32)
        nonempty = valid_counts > 0
        denominators = np.maximum(valid_counts, 1).astype(np.float32)[:, None]
        window_means = (
            np.sum(feature_window, axis=0, where=valid_window[:, :, None], dtype=np.float64)
            .astype(np.float32) / denominators
        )
        window_means[~nonempty] = feature_mean
        filled_window = np.where(valid_window[:, :, None], feature_window, window_means[None, :, :])
        standardized = (filled_window - feature_mean[None, None, :]) / feature_std[None, None, :]
        mask_channel = valid_window[:, :, None].astype(np.float32)
        model_inputs = np.concatenate([standardized, mask_channel], axis=2).transpose(1, 2, 0)
        coverage = (valid_counts.astype(np.float32) / feature_window.shape[0])[:, None]
        return (
            np.ascontiguousarray(model_inputs, dtype=np.float32),
            np.ascontiguousarray(coverage, dtype=np.float32),
            nonempty,
        )

    def predict_tcn(time_idx, stock_indices, model):
        stock_indices = np.asarray(stock_indices, dtype=np.int64)
        predictions = np.full(stock_indices.size, 0.5, dtype=np.float32)
        model.eval()
        for batch_start in range(0, stock_indices.size, TCN_BATCH):
            batch_stop = min(batch_start + TCN_BATCH, stock_indices.size)
            batch_stocks = stock_indices[batch_start:batch_stop]
            batch_inputs, batch_coverage, batch_nonempty = build_model_batch(time_idx, batch_stocks)
            if not np.any(batch_nonempty):
                continue
            usable = np.flatnonzero(batch_nonempty)
            input_tensor = torch.from_numpy(batch_inputs[usable]).to(DEVICE, non_blocking=True)
            coverage_tensor = torch.from_numpy(batch_coverage[usable]).to(DEVICE, non_blocking=True)
            with torch.inference_mode():
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                    batch_predictions = model(input_tensor, coverage_tensor)
            predictions[batch_start + usable] = (
                batch_predictions.squeeze(1).float().cpu().numpy()
            )
        return predictions

    def train_one_tcn(segments, rounds_info):
        model = WindowTCN(
            MODEL_INPUT_CHANNELS, HIDDEN_CHANNELS, DILATIONS, DROPOUT
        ).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
        loss_function = nn.MSELoss()
        gradient_scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
        training_rng = np.random.default_rng(TCN_SEED)
        train_start_t = int(segments[0][1])
        train_stop_t = int(segments[-1][2])
        time_bins = np.array_split(
            np.arange(train_start_t, train_stop_t, dtype=np.int64), TRAIN_TIME_BINS
        )
        for epoch_idx in range(MAX_EPOCHS):
            model.train()
            selected_times = []
            for time_bin in time_bins:
                sample_count = min(TIMES_PER_BIN, time_bin.size)
                selected_times.extend(
                    training_rng.choice(time_bin, size=sample_count, replace=False).tolist()
                )
            training_rng.shuffle(selected_times)
            for time_idx in selected_times:
                time_idx = int(time_idx)
                eligible = labeled_stock_indices(time_idx)
                if eligible.size == 0:
                    continue
                if eligible.size > STOCKS_PER_TIME:
                    eligible = training_rng.choice(eligible, size=STOCKS_PER_TIME, replace=False)
                training_rng.shuffle(eligible)
                for batch_start in range(0, eligible.size, TCN_BATCH):
                    batch_stop = min(batch_start + TCN_BATCH, eligible.size)
                    batch_stocks = eligible[batch_start:batch_stop]
                    batch_inputs, batch_coverage, batch_nonempty = build_model_batch(time_idx, batch_stocks)
                    if not np.all(batch_nonempty):
                        usable = np.flatnonzero(batch_nonempty)
                        batch_stocks = batch_stocks[usable]
                        batch_inputs = batch_inputs[usable]
                        batch_coverage = batch_coverage[usable]
                    if batch_stocks.size == 0:
                        continue
                    batch_targets = Y1_TCN[time_idx, batch_stocks].astype(np.float32)[:, None]
                    input_tensor = torch.from_numpy(batch_inputs).to(DEVICE, non_blocking=True)
                    coverage_tensor = torch.from_numpy(batch_coverage).to(DEVICE, non_blocking=True)
                    target_tensor = torch.from_numpy(batch_targets).to(DEVICE, non_blocking=True)
                    optimizer.zero_grad(set_to_none=True)
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                        prediction_tensor = model(input_tensor, coverage_tensor)
                        loss = loss_function(prediction_tensor, target_tensor)
                    gradient_scaler.scale(loss).backward()
                    gradient_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    gradient_scaler.step(optimizer)
                    gradient_scaler.update()
            scheduler.step()
        return model

    # 每折 TCN 预测
    tcn_fold_ranks = {}
    tcn_fold_metrics = {}
    for fold in ACTIVE_FOLDS:
        recent_start = max(fold.train_start, fold.train_stop - ACTIVE_LOOKBACK)
        cache_path = RUNTIME_CACHE_DIR / f"{fold.name}_{RUN_FINGERPRINT}_tcn_pred.npy"
        if cache_path.exists():
            tcn_fold_pred = np.load(cache_path)
        else:
            tcn_model = train_one_tcn([("train", recent_start, fold.train_stop)], None)
            rows, groups = row_slice_for_times("train", fold.valid_start, fold.valid_stop)
            times = np.asarray(common["train"]["time"][rows], dtype=np.int64)
            stocks = np.asarray(common["train"]["stock"][rows], dtype=np.int64)
            tcn_fold_pred = np.empty(times.size, dtype=np.float32)
            for time_idx in np.unique(times):
                positions = np.flatnonzero(times == time_idx)
                row_stocks = stocks[positions]
                sorted_stocks = np.sort(row_stocks)
                preds = predict_tcn(int(time_idx), sorted_stocks, tcn_model)
                tcn_fold_pred[positions] = preds[np.searchsorted(sorted_stocks, row_stocks)]
            del tcn_model
            gc.collect()
            atomic_save_npy(cache_path, tcn_fold_pred)
        valid_groups = fold_data[fold.name]["valid_groups"]
        valid_target = fold_data[fold.name]["valid_target"]
        tcn_fold_ranks[fold.name] = group_rank_transform(tcn_fold_pred, valid_groups)
        tcn_fold_metrics[fold.name] = score_prediction(
            tcn_fold_ranks[fold.name], valid_target, valid_groups
        )

    # TCN 权重网格：作用于已选锚点+专家融合
    main_blend_rank = {
        fold_name: (1.0 - selected_recent_weight) * fd["anchor_rank"]
        + selected_recent_weight * fd["expert_rank"][(SELECTED_FEATURE_SET, SELECTED_STRATEGY, selected_expert_rounds)]
        for fold_name, fd in fold_data.items()
    }
    tcn_search_rows = []
    for w_t in TCN_WEIGHTS[1:]:
        deltas = []
        for fold_name, fd in fold_data.items():
            blended = (1.0 - w_t) * main_blend_rank[fold_name] + w_t * tcn_fold_ranks[fold_name]
            m = score_prediction(blended, fd["valid_target"], fd["valid_groups"])
            base_m = score_prediction(main_blend_rank[fold_name], fd["valid_target"], fd["valid_groups"])
            deltas.append(m["mean_rank_ic"] - base_m["mean_rank_ic"])
        tcn_search_rows.append({
            "tcn_weight": w_t,
            "mean_delta": float(np.mean(deltas)),
            "positive_fold_count": int(np.sum(np.asarray(deltas) > 0)),
        })
    tcn_search = pd.DataFrame(tcn_search_rows)
    atomic_write_csv(RUN_DIR / "tcn_weight_search.csv", tcn_search)
    display(tcn_search)
    positive = tcn_search[tcn_search["positive_fold_count"] >= MIN_POSITIVE_FOLDS]
    if positive.empty:
        tcn_weight = 0.0
        print("TCN：没有候选达到 ≥2 折为正，权重回退 0。")
    else:
        tcn_weight = float(positive.sort_values("mean_delta", ascending=False).iloc[0]["tcn_weight"])
        print("TCN 权重选择：", tcn_weight)

    if tcn_weight > 0.0:
        final_recent_start = VALID_STOP - RECENT_LOOKBACK
        tcn_cache_path = RUNTIME_CACHE_DIR / f"tcn_test_{RUN_FINGERPRINT}.npy"
        if tcn_cache_path.exists():
            tcn_test_pred = np.load(tcn_cache_path)
        else:
            tcn_final_model = train_one_tcn(
                [("train", final_recent_start, TRAIN_STOP), ("valid", VALID_START, VALID_STOP)], None
            )
            test_rows, test_groups = row_slice_for_times("test", TEST_START, TEST_STOP)
            times = np.asarray(common["test"]["time"], dtype=np.int64)
            stocks = np.asarray(common["test"]["stock"], dtype=np.int64)
            tcn_test_pred = np.empty(times.size, dtype=np.float32)
            for time_idx in np.unique(times):
                positions = np.flatnonzero(times == time_idx)
                row_stocks = stocks[positions]
                sorted_stocks = np.sort(row_stocks)
                preds = predict_tcn(int(time_idx), sorted_stocks, tcn_final_model)
                tcn_test_pred[positions] = preds[np.searchsorted(sorted_stocks, row_stocks)]
            del tcn_final_model
            gc.collect()
            atomic_save_npy(tcn_cache_path, tcn_test_pred)
        tcn_grid = np.full((TEST_TIME_POINTS, STOCK_COUNT), 0.5, dtype=np.float32)
        offset = 0
        for local_time, size in enumerate(np.asarray(common["test"]["groups"], dtype=np.int32)):
            size = int(size)
            stocks = np.asarray(common["test"]["stock"][offset:offset + size], dtype=np.int32)
            tcn_grid[local_time, stocks] = group_rank_transform(
                tcn_test_pred[offset:offset + size],
                np.array([size], dtype=np.int32),
            )
            offset += size
else:
    print("TCN 未启用（DSCR_EXP010_USE_TCN=0），仅使用锚点+专家融合。")


TCN 未启用（DSCR_EXP010_USE_TCN=0），仅使用锚点+专家融合。


### 11. Train+Valid 近期专家重训与完整 Test 预测

Test 锚点直接读取正式提交。专家只训练一次；预测先逐时间截面排名，再按选定权重融合并再次排名。即使 `promoted=False`，本实验仍保存自己的 `prediction.npy`，但不会改动正式提交。

In [13]:
anchor_grid = np.load(ANCHOR_PATH)
assert anchor_grid.shape == (TEST_TIME_POINTS, STOCK_COUNT)
assert anchor_grid.dtype == np.float32
assert np.isfinite(anchor_grid).all()

if RUN_MODE == "preflight":
    expert_grid = anchor_grid.copy()
    prediction_grid = anchor_grid.copy()
    test_anchor_correlation = 1.0
    final_model_reused = False
    print("Preflight：复制正式锚点用于验证完整结果写出。")
else:
    import lightgbm as lgb

    final_recent_start = VALID_STOP - RECENT_LOOKBACK
    final_model_path = RUN_DIR / "model_recent.txt"
    final_model_metadata_path = RUN_DIR / "model_recent.metadata.json"
    final_model_fingerprint = hashlib.sha256(
        json.dumps({
            "run_fingerprint": RUN_FINGERPRINT,
            "train_start": final_recent_start,
            "train_stop": VALID_STOP,
            "rounds": selected_expert_rounds,
            "feature_set": SELECTED_FEATURE_SET,
            "strategy": SELECTED_STRATEGY,
        }, sort_keys=True).encode("utf-8")
    ).hexdigest()

    final_model_reused = False
    if final_model_path.exists() and final_model_metadata_path.exists():
        saved_model_metadata = json.loads(final_model_metadata_path.read_text(encoding="utf-8"))
        if saved_model_metadata.get("fingerprint") == final_model_fingerprint:
            recent_final_model = lgb.Booster(model_str=final_model_path.read_text(encoding="utf-8"))
            final_model_reused = True
            print("复用最终近期专家模型。")

    if not final_model_reused:
        print(
            f"训练最终近期专家 [{final_recent_start}, {VALID_STOP}) "
            f"{SELECTED_FEATURE_SET}/{SELECTED_STRATEGY}/r{selected_expert_rounds}。",
            flush=True,
        )
        recent_final_model, final_train_info = train_ranker(
            [
                ("train", final_recent_start, TRAIN_STOP),
                ("valid", VALID_START, VALID_STOP),
            ],
            selected_expert_rounds,
            SELECTED_FEATURE_SET,
            SELECTED_STRATEGY,
        )
        save_model_text_atomic(recent_final_model, final_model_path)
        atomic_write_json(final_model_metadata_path, {
            "fingerprint": final_model_fingerprint,
            "train_start": final_recent_start,
            "train_stop": VALID_STOP,
            "rounds": selected_expert_rounds,
            "feature_set": SELECTED_FEATURE_SET,
            "strategy": SELECTED_STRATEGY,
            **final_train_info,
        })

    raw_test_cache_path = RUNTIME_CACHE_DIR / f"raw_test_{final_model_fingerprint[:16]}.npy"
    if raw_test_cache_path.exists():
        raw_test_prediction = np.load(raw_test_cache_path)
        assert raw_test_prediction.shape == (int(common["test"]["groups"].sum()),)
        print("复用近期专家 Test 原始预测。")
    else:
        raw_test_prediction, test_groups = predict_interval(
            recent_final_model,
            "test",
            TEST_START,
            TEST_STOP,
            selected_expert_rounds,
            SELECTED_FEATURE_SET,
        )
        atomic_save_npy(raw_test_cache_path, raw_test_prediction)

    test_groups = np.asarray(common["test"]["groups"], dtype=np.int32)
    test_stocks = np.asarray(common["test"]["stock"], dtype=np.int32)
    expert_grid = np.full((TEST_TIME_POINTS, STOCK_COUNT), 0.5, dtype=np.float32)
    prediction_grid = np.full((TEST_TIME_POINTS, STOCK_COUNT), 0.5, dtype=np.float32)

    offset = 0
    for local_time, size in enumerate(test_groups):
        size = int(size)
        stocks = test_stocks[offset:offset + size]
        expert_rank = rankdata(
            raw_test_prediction[offset:offset + size], method="average"
        ).astype(np.float32) / float(size)
        anchor_rank = rankdata(
            anchor_grid[local_time, stocks], method="average"
        ).astype(np.float32) / float(size)
        combined = (
            (1.0 - selected_recent_weight) * anchor_rank
            + selected_recent_weight * expert_rank
        )
        blended_rank = rankdata(combined, method="average").astype(np.float32) / float(size)
        if tcn_weight > 0.0 and tcn_grid is not None:
            tcn_rank = tcn_grid[local_time, stocks]
            combined = (1.0 - tcn_weight) * blended_rank + tcn_weight * tcn_rank
            blended_rank = rankdata(combined, method="average").astype(np.float32) / float(size)
        expert_grid[local_time, stocks] = expert_rank
        prediction_grid[local_time, stocks] = blended_rank
        offset += size
    assert offset == raw_test_prediction.size == int(test_groups.sum())

    if selected_recent_weight == 0.0 and tcn_weight == 0.0:
        prediction_grid = anchor_grid.copy()

    test_anchor_correlation = mean_cross_sectional_rank_correlation(
        prediction_grid, anchor_grid, test_groups, test_stocks
    )

test_mask = np.zeros((TEST_TIME_POINTS, STOCK_COUNT), dtype=bool)
test_mask[
    np.asarray(common["test"]["time"], dtype=np.int32) - TEST_START,
    np.asarray(common["test"]["stock"], dtype=np.int32),
] = True

assert int(test_mask.sum()) == 2_042_538
assert prediction_grid.shape == expert_grid.shape == (TEST_TIME_POINTS, STOCK_COUNT)
assert prediction_grid.dtype == expert_grid.dtype == np.float32
assert np.isfinite(prediction_grid).all() and np.isfinite(expert_grid).all()
assert np.all(prediction_grid[~test_mask] == 0.5)
assert np.all(expert_grid[~test_mask] == 0.5)
assert float(prediction_grid.min()) >= 0.0 and float(prediction_grid.max()) <= 1.0

atomic_save_npy(RUN_DIR / "expert_prediction.npy", expert_grid)
atomic_save_npy(RUN_DIR / "prediction.npy", prediction_grid)

print(pd.Series({
    "prediction_shape": prediction_grid.shape,
    "prediction_dtype": str(prediction_grid.dtype),
    "evaluation_count": int(test_mask.sum()),
    "non_evaluation_count": int((~test_mask).sum()),
    "non_evaluation_all_0_5": bool(np.all(prediction_grid[~test_mask] == 0.5)),
    "minimum": float(prediction_grid.min()),
    "maximum": float(prediction_grid.max()),
    "mean": float(prediction_grid.mean()),
    "test_anchor_rank_correlation": test_anchor_correlation,
    "tcn_weight": tcn_weight,
}))


训练最终近期专家 [1459, 3161) enhanced/decay_1200/r8。
prediction_shape                (442, 5282)
prediction_dtype                    float32
evaluation_count                    2042538
non_evaluation_count                 292106
non_evaluation_all_0_5                 True
minimum                            0.000629
maximum                                 1.0
mean                                    0.5
test_anchor_rank_correlation            1.0
tcn_weight                              0.0
dtype: object


## Checks

### 12. 保存统一结果并复读验收

In [14]:
prediction_path = RUN_DIR / "prediction.npy"
expert_prediction_path = RUN_DIR / "expert_prediction.npy"
valid_prediction_path = RUN_DIR / "valid_prediction.npy"

loaded_prediction = np.load(prediction_path, mmap_mode="r")
assert loaded_prediction.shape == (TEST_TIME_POINTS, STOCK_COUNT)
assert loaded_prediction.dtype == np.float32
assert np.isfinite(loaded_prediction).all()
assert np.all(loaded_prediction[~test_mask] == 0.5)

test_correlation_gate_passed = bool(test_anchor_correlation >= MIN_TEST_ANCHOR_CORRELATION)
if RUN_MODE == "full" and not test_correlation_gate_passed:
    promoted = False

existing_metrics_path = RUN_DIR / "metrics.json"
existing_metadata_path = RUN_DIR / "metadata.json"
existing_metrics = (
    json.loads(existing_metrics_path.read_text(encoding="utf-8"))
    if existing_metrics_path.exists()
    else {}
)
existing_metadata = (
    json.loads(existing_metadata_path.read_text(encoding="utf-8"))
    if existing_metadata_path.exists()
    else {}
)
recorded_online_rank_ic = existing_metrics.get(
    "online_rank_ic", existing_metadata.get("online_rank_ic")
)

status = (
    "preflight_only"
    if RUN_MODE == "preflight"
    else (
        "submitted_online_best"
        if recorded_online_rank_ic is not None
        else ("completed_candidate" if promoted else "completed_not_promoted")
    )
)

selected_internal_rows = weight_search[weight_search["selected"]]
selected_internal_metrics = (
    selected_internal_rows.iloc[0].to_dict()
    if not selected_internal_rows.empty
    else {}
)

metrics = {
    "run_mode": RUN_MODE,
    "status": status,
    "selected_feature_set": SELECTED_FEATURE_SET,
    "selected_strategy": SELECTED_STRATEGY,
    "selected_expert_rounds": selected_expert_rounds,
    "selected_recent_weight": selected_recent_weight,
    "internal_selection": selected_internal_metrics,
    "feature_diagnosis": json.loads(diag_df.to_json(orient="records")),
    "feature_enhanced_kept": feature_enhanced_kept,
    "time_strategy": json.loads(strat_df.to_json(orient="records")),
    "strategy_decay_selected": decay_selected,
    "selected_plateau_span": selected_plateau_span,
    "plateau_passed": plateau_passed,
    "official_valid_anchor": official_base_metrics,
    "official_valid_expert": official_expert_metrics,
    "official_valid_blend": official_blend_metrics,
    "official_mean_delta": official_mean_delta,
    "official_late_delta": official_late_delta,
    "official_worst_quarter_delta": official_worst_delta,
    "anchor_reproduced": official_anchor_reproduced,
    "exp009_reproduced": exp009_reproduced,
    "test_anchor_rank_correlation": test_anchor_correlation,
    "test_correlation_gate_passed": test_correlation_gate_passed,
    "tcn_weight": tcn_weight,
    "promoted": promoted,
}


def json_safe(value):
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


metrics = json_safe(metrics)
if recorded_online_rank_ic is not None:
    metrics.update({
        "online_rank_ic": float(recorded_online_rank_ic),
        "online_result_source": existing_metrics.get(
            "online_result_source", "preserved_existing_record"
        ),
    })

prediction_sha256 = file_sha256(prediction_path)
expert_sha256 = file_sha256(expert_prediction_path)
metadata = {
    "experiment_id": EXPERIMENT_ID,
    "run_mode": RUN_MODE,
    "status": status,
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_fingerprint": RUN_FINGERPRINT,
    "manifest_sha256": manifest_sha256,
    "anchor_path": str(ANCHOR_PATH),
    "anchor_sha256": file_sha256(ANCHOR_PATH),
    "prediction_path": str(prediction_path),
    "prediction_sha256": prediction_sha256,
    "expert_prediction_sha256": expert_sha256,
    "shape": [TEST_TIME_POINTS, STOCK_COUNT],
    "dtype": "float32",
    "finite": True,
    "evaluation_count": int(test_mask.sum()),
    "non_evaluation_count": int((~test_mask).sum()),
    "non_evaluation_value": 0.5,
    "minimum": float(loaded_prediction.min()),
    "maximum": float(loaded_prediction.max()),
    "mean": float(loaded_prediction.mean()),
    "base_features": FEATURE_STOP,
    "incremental_features": INCR_SIZE,
    "selected_feature_set": SELECTED_FEATURE_SET,
    "selected_strategy": SELECTED_STRATEGY,
    "selected_expert_rounds": selected_expert_rounds,
    "selected_recent_weight": selected_recent_weight,
    "selection_reason": selection_reason,
    "tcn_weight": tcn_weight,
    "promoted": promoted,
    "formal_submission_overwritten": False,
}
if recorded_online_rank_ic is not None:
    metadata.update({
        "online_rank_ic": float(recorded_online_rank_ic),
        "online_status": existing_metadata.get("online_status", "submitted"),
        "online_recorded_at": existing_metadata.get("online_recorded_at"),
    })

def fmt_ic(value: float) -> str:
    return f"{float(value):.6f}"

gate_rows = [
    ("w > 0", selected_recent_weight > 0.0, f"{selected_recent_weight:.2f}"),
    ("锚点复现 |valid_anchor - 0.09294016824452567| <= 0.0003", official_anchor_reproduced,
     f"{abs(official_base_metrics['mean_rank_ic'] - ANCHOR_EXPECTED_VALID_IC):.6f}"),
    ("official_mean_delta >= 0.0003", official_mean_delta >= OFFICIAL_MIN_IMPROVEMENT,
     f"{official_mean_delta:+.6f}"),
    ("official_late_delta >= -0.0002", official_late_delta >= -OFFICIAL_MAX_LATE_DROP,
     f"{official_late_delta:+.6f}"),
    ("official_worst_quarter_delta >= -0.0015", official_worst_delta >= -OFFICIAL_MAX_WORST_QUARTER_DROP,
     f"{official_worst_delta:+.6f}"),
    ("test_anchor_correlation >= 0.97", test_correlation_gate_passed,
     f"{test_anchor_correlation:.6f}"),
    ("稳定性平台 plateau_span < 0.001", plateau_passed, f"{selected_plateau_span:.6f}"),
]
gate_df = pd.DataFrame(
    [{"gate": name, "passed": passed, "value": value} for name, passed, value in gate_rows]
)
atomic_write_csv(RUN_DIR / "promotion_gates.csv", gate_df)
display(gate_df)

report_lines = [
    "# 锚点增强融合实验报告（exp_010）",
    "",
    f"- 运行模式：`{RUN_MODE}`",
    f"- 状态：`{status}`",
    f"- 主特征：`legacy_328`（{FEATURE_STOP} 维）",
    f"- 增量特征：`exp_008 surprise/trend/history_state`（{INCR_SIZE} 维）",
    f"- 已选特征集：`{SELECTED_FEATURE_SET}`（{'启用增强' if feature_enhanced_kept else '回退 328'}）",
    f"- 已选时间策略：`{SELECTED_STRATEGY}`（{'decay_1200' if decay_selected else 'full'}）",
    f"- 已选专家轮数：`{selected_expert_rounds}`",
    f"- 已选融合权重：`{selected_recent_weight:.2f}`",
    f"- TCN 权重：`{tcn_weight:.2f}`（{'启用' if tcn_weight > 0 else '未启用'}）",
    f"- 锚点：`{ANCHOR_PATH}`（只读，未覆盖）",
    f"- 官方 Valid 锚点 RankIC：`{fmt_ic(official_base_metrics['mean_rank_ic'])}`",
    f"- 官方 Valid 专家 RankIC：`{fmt_ic(official_expert_metrics['mean_rank_ic'])}`",
    f"- 官方 Valid 融合 RankIC：`{fmt_ic(official_blend_metrics['mean_rank_ic'])}`",
    f"- 官方 Valid 增量：`{official_mean_delta:+.6f}`（后段 `{official_late_delta:+.6f}`，最差季度 `{official_worst_delta:+.6f}`）",
    f"- 稳定性平台 span：`{selected_plateau_span:.6f}`",
    f"- Test 与正式锚点平均截面秩相关：`{test_anchor_correlation:.6f}`",
    f"- 是否晋级：`{promoted}`",
    f"- 预测 SHA-256：`{prediction_sha256}`",
    "",
    "## 1. exp_009 基线复现（Step A）",
    "",
    f"- 折内 (16, 0.25) 融合与 8 轮锚点相对 exp_009 最大绝对偏差：`{max(repro_df['anchor_diff'].abs().max(), repro_df['blend_diff'].abs().max()) if RUN_MODE == 'full' else 0.0:.6f}`（容差 5e-4）",
    f"- 复现结论：`{'通过' if exp009_reproduced else '未通过'}`",
    "",
    "## 2. 特征诊断（Step B）",
    "",
    "- 比较纯专家（不融合）在 16 轮的折内平均 RankIC：",
    "",
]
for _, row in diag_df.iterrows():
    report_lines.append(
        f"- `{row['fold']}`：legacy `{row['legacy_ic']:.6f}` → enhanced `{row['enhanced_ic']:.6f}`，增量 `{row['delta']:+.6f}`"
    )
report_lines.extend([
    "",
    f"- 3 折均值增量：`{diag_mean_gain:+.6f}`；是否 3 折均正：`{diag_all_positive}`；门槛：`{DIAGNOSIS_MIN_MEAN_GAIN}`",
    f"- 判定：`{'启用增强 433' if feature_enhanced_kept else '回退 legacy_328'}`",
    "",
    "## 3. 时间策略（Step C）",
    "",
])
for _, row in strat_df.iterrows():
    report_lines.append(
        f"- `{row['fold']}`：full `{row['full_mean_ic']:.6f}` / late `{row['full_late_ic']:.6f}` → decay `{row['decay_mean_ic']:.6f}` / late `{row['decay_late_ic']:.6f}`"
    )
report_lines.extend([
    "",
    f"- decay 相对 full 的 3 折均值增量：`{strat_mean_delta:+.6f}`；后段增量：`{strat_late_delta:+.6f}`",
    f"- 判定：`{SELECTED_STRATEGY}`",
    "",
    "## 4. 轮数与权重选择（Step D）",
    "",
    selection_reason,
    f"- 已选组合：{SELECTED_FEATURE_SET}/{SELECTED_STRATEGY}、{selected_expert_rounds} 轮、w={selected_recent_weight:.2f}",
    "",
    "## 5. 官方 Valid 一次性晋级检查",
    "",
    "- 锚点（legacy_328 / full / 8 轮）："
    f" mean `{fmt_ic(official_base_metrics['mean_rank_ic'])}`"
    f" | late `{fmt_ic(official_base_metrics['late_half_rank_ic'])}`"
    f" | worst_q `{fmt_ic(official_base_metrics['worst_quarter_rank_ic'])}`",
    "- 近期专家："
    f" mean `{fmt_ic(official_expert_metrics['mean_rank_ic'])}`"
    f" | late `{fmt_ic(official_expert_metrics['late_half_rank_ic'])}`"
    f" | worst_q `{fmt_ic(official_expert_metrics['worst_quarter_rank_ic'])}`",
    "- 已选融合："
    f" mean `{fmt_ic(official_blend_metrics['mean_rank_ic'])}`"
    f" | late `{fmt_ic(official_blend_metrics['late_half_rank_ic'])}`"
    f" | worst_q `{fmt_ic(official_blend_metrics['worst_quarter_rank_ic'])}`",
    "",
    "### 门槛逐项判定",
    "",
])
for name, passed, value in gate_rows:
    report_lines.append(f"- [{'x' if passed else ' '}] `{name}` → {value}")
report_lines.extend([
    "",
    f"**晋级结论：`{promoted}`**",
    "",
    "## 6. 与历史实验对比",
    "",
    "| 实验 | 官方 Valid RankIC | 线上 RankIC | 说明 |",
    "| --- | --- | --- | --- |",
    "| exp_003（正式提交/锚点） | 0.092940 | 0.108105 | legacy_328 + LambdaRank 8 轮 |",
    "| exp_007 | 0.094446 | 0.109959 | 锚点+近期专家 rank 融合 0.65/0.35 |",
    "| exp_008 | 0.084385 | 0.101942 | 完整因果补全 + 363 特征 + decay |",
    f"| exp_009 | 0.093615 | 0.109928 | 锚点+近期专家 0.75/0.25，16 轮 |",
    f"| exp_010（本实验） | `{fmt_ic(official_blend_metrics['mean_rank_ic'])}` | 待提交 | {SELECTED_FEATURE_SET}/{SELECTED_STRATEGY}/r{selected_expert_rounds}/w{selected_recent_weight:.2f}" + (" + TCN" if tcn_weight > 0 else "") + " |",
    "",
    "## 产物",
    "",
    "- `prediction.npy`：本实验完整 Test 结果（晋级时另存 `promoted_candidate.npy`）。",
    "- `expert_prediction.npy`：专家逐时间排名结果。",
    "- `valid_prediction.npy`：官方 Valid 已选融合结果。",
    "- `fold_results.csv` / `blend_fold_results.csv`：折内明细。",
    "- `feature_diagnosis.csv` / `time_strategy.csv`：特征与时间策略诊断。",
    "- `weight_search.csv`：轮数/权重稳定性门槛与最终选择。",
    "- `official_valid_results.csv` / `promotion_gates.csv`：官方 Valid 一次性检查。",
    "- `exp009_reproduction.csv`：exp_009 基线复现核对。",
    "",
    "> 本实验不会自动覆盖 `04_results/final_submission/prediction.npy`。",
])
if recorded_online_rank_ic is not None:
    report_lines.extend([
        "",
        "## 已记录线上结果",
        "",
        f"- 线上 RankIC：`{float(recorded_online_rank_ic):.6f}`",
        "- 重新运行 Notebook 时会保留该外部记录。",
    ])

atomic_write_json(RUN_DIR / "metrics.json", metrics)
atomic_write_json(RUN_DIR / "metadata.json", metadata)
atomic_write_text(RUN_DIR / "experiment_report.md", chr(10).join(report_lines) + chr(10))

if RUN_MODE == "full" and promoted:
    atomic_save_npy(RUN_DIR / "promoted_candidate.npy", prediction_grid)

required_outputs = [
    RUN_DIR / "prediction.npy",
    RUN_DIR / "expert_prediction.npy",
    RUN_DIR / "valid_prediction.npy",
    RUN_DIR / "fold_results.csv",
    RUN_DIR / "blend_fold_results.csv",
    RUN_DIR / "feature_diagnosis.csv",
    RUN_DIR / "time_strategy.csv",
    RUN_DIR / "weight_search.csv",
    RUN_DIR / "official_valid_results.csv",
    RUN_DIR / "promotion_gates.csv",
    RUN_DIR / "metrics.json",
    RUN_DIR / "metadata.json",
    RUN_DIR / "experiment_report.md",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError("结果文件不完整：" + str(missing_outputs))

print("实验运行完成，结果目录：", RUN_DIR)
print("prediction.npy：", prediction_path)
print("状态：", status)


,gate,passed,value
0,w > 0,False,0.00
1,锚点复现 |valid_anchor - 0.09294016824452567| <= 0...,True,0.000000
2,official_mean_delta >= 0.0003,False,+0.000000
3,official_late_delta >= -0.0002,True,+0.000000
4,official_worst_quarter_delta >= -0.0015,True,+0.000000
5,test_anchor_correlation >= 0.97,True,1.000000
6,稳定性平台 plateau_span < 0.001,True,0.000000


实验运行完成，结果目录： D:\google_dl\book\友安杯\04_results\exp_010_anchor_enhanced_blend
prediction.npy： D:\google_dl\book\友安杯\04_results\exp_010_anchor_enhanced_blend\prediction.npy
状态： completed_not_promoted


## Takeaways

运行完成后，以底部代码单元输出和 `experiment_report.md` 为准：

- `completed_candidate`：候选通过内部稳定性、官方 Valid、稳定性平台与 Test 锚点相关性门槛，可进入人工提交评估。
- `completed_not_promoted`：结果文件完整生成，但不建议替换正式提交。
- `preflight_only`：仅证明数据契约、特征银行对齐、选择逻辑和文件写出可用，不代表模型成绩。
